# 02 · Synthetic Data Pipeline

Walks through the renderer:
1. Gnomonic projection (the right projection for small fields).
2. Star intensity / PSF size as a function of magnitude.
3. Rendered samples at different field widths, noise levels, rotations.
4. Augmentation grid (note: full 180° rotations are correct here).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
from src.data.catalog import load_hyg_catalog
from src.data.renderer import render_star_field, sample_random_pointing
from src.utils.coordinates import gnomonic_project
catalog = load_hyg_catalog(ROOT / 'data/catalogs/hygdata_v3.csv', mag_limit=8.0)

## Gnomonic projection demo
Plot a 1° grid before and after projection at different declinations to see the warp.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, dec0 in zip(axes, [0.0, 45.0, 80.0]):
    ras, decs = np.meshgrid(np.arange(-30, 31, 5), np.arange(-30, 31, 5))
    xs, ys = gnomonic_project(ras + 180.0, decs + dec0, 180.0, dec0)
    ax.plot(xs, ys, color='#7c5cff', linewidth=0.8)
    ax.plot(xs.T, ys.T, color='#7c5cff', linewidth=0.8)
    ax.set_title(f'Tangent plane @ Dec={dec0:.0f}°'); ax.set_aspect('equal')
fig.tight_layout(); fig.savefig(ROOT / 'reports/figures/02_gnomonic_grid.png', dpi=150)
plt.show()

## Render samples at different field widths

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4), facecolor='#0b0d17')
rng = np.random.default_rng(0)
for ax, fw in zip(axes, [15, 30, 50, 80]):
    img = render_star_field(ra_center=85, dec_center=-2, field_width=fw, rotation=0,
                            catalog=catalog, image_size=224, rng=np.random.default_rng(int(fw)))
    ax.imshow(img); ax.set_title(f'FOV {fw}°', color='white'); ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout(); fig.savefig(ROOT / 'reports/figures/02_fov_grid.png', dpi=150)
plt.show()

## Augmentation grid

In [ ]:
from PIL import Image
from src.data.augmentations import build_train_transforms
tfm = build_train_transforms(224)
base = render_star_field(85, -2, 30, 0, catalog, image_size=224, rng=np.random.default_rng(7))
base_pil = Image.fromarray((base * 255).astype(np.uint8))
fig, axes = plt.subplots(2, 4, figsize=(12, 6), facecolor='#0b0d17')
for ax in axes.flatten():
    t = tfm(base_pil)
    # de-normalize for display
    img = (t.permute(1, 2, 0).numpy() * np.array([0.15, 0.15, 0.20]) + np.array([0.10, 0.10, 0.15]))
    ax.imshow(np.clip(img, 0, 1)); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Augmentation samples (180° rotation, flips, jitter, blur)', color='white')
fig.tight_layout(); fig.savefig(ROOT / 'reports/figures/02_augmentation_grid.png', dpi=150)
plt.show()